# This script scrapes match data from understatapi and saves it to a CSV file.

In [ ]:
import os

import pandas as pd
from understatapi import UnderstatClient
import time
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

### Inspect raw data from UnderStat

In [2]:
def inspect_raw_data(league: str, season: str, sample_size: int = 1):
    """
    Inspect the raw data structure from understatapi to understand columns and format.
    
    Args:
        league (str): The league code (e.g., 'EPL')
        season (str): The season year (e.g., '2025')
        sample_size (int): Number of matches to inspect
    """
    with UnderstatClient() as understat:
        fixtures = understat.league(league=league).get_match_data(season=season)
        match_ids = [int(match['id']) for match in fixtures if match.get('isResult') is True]
        print(len(match_ids))
        
        print(f"Inspecting {sample_size} match(es) from {league} {season}...")
        
        for idx, match_id in enumerate(match_ids[:sample_size]):
            try:
                match_endpoint = understat.match(match=str(match_id))
                raw_shots = match_endpoint.get_shot_data(timeout=30)
                raw_context = match_endpoint.get_match_info(timeout=30)
                
                print(f"\n{'='*10}")
                print(f"Match ID: {match_id}")
                print(f"{'='*10}")
                
                print("\n--- RAW SHOTS DATA ---")
                print(f"Type: {type(raw_shots)}")
                if isinstance(raw_shots, list) and len(raw_shots) > 0:
                    print(f"Number of shots: {len(raw_shots)}")
                    print(f"First shot keys: {raw_shots[0].keys()}")
                    print(f"Sample shot:\n{json.dumps(raw_shots[0], indent=2)}")
                
                print("\n--- RAW CONTEXT DATA ---")
                print(f"Type: {type(raw_context)}")
                if isinstance(raw_context, dict):
                    print(f"Keys: {raw_context.keys()}")
                    print(f"Sample context:\n{json.dumps(raw_context, indent=2, default=str)}")
                    
            except Exception as exc:
                print(f"\n[Warning] Could not inspect Match ID {match_id}: {exc}")
            
            if idx < sample_size - 1:
                time.sleep(0.4)

inspect_raw_data(league='EPL', season='2025', sample_size=2)

370
Inspecting 2 match(es) from EPL 2025...

Match ID: 28778

--- RAW SHOTS DATA ---
Type: <class 'dict'>

--- RAW CONTEXT DATA ---
Type: <class 'dict'>
Keys: dict_keys(['id', 'fid', 'h', 'a', 'date', 'league_id', 'season', 'h_goals', 'a_goals', 'team_h', 'team_a', 'h_xg', 'a_xg', 'h_w', 'h_d', 'h_l', 'league', 'h_shot', 'a_shot', 'h_shotOnTarget', 'a_shotOnTarget', 'h_deep', 'a_deep', 'a_ppda', 'h_ppda', 'isData'])

[Warning] Could not inspect Match ID 28778: name 'json' is not defined

Match ID: 28779

--- RAW SHOTS DATA ---
Type: <class 'dict'>

--- RAW CONTEXT DATA ---
Type: <class 'dict'>
Keys: dict_keys(['id', 'fid', 'h', 'a', 'date', 'league_id', 'season', 'h_goals', 'a_goals', 'team_h', 'team_a', 'h_xg', 'a_xg', 'h_w', 'h_d', 'h_l', 'league', 'h_shot', 'a_shot', 'h_shotOnTarget', 'a_shotOnTarget', 'h_deep', 'a_deep', 'a_ppda', 'h_ppda', 'isData'])

[Warning] Could not inspect Match ID 28779: name 'json' is not defined


### Scrape match data and save the raw data to per-match JSON files.

In [4]:
import json


def scrape_match_data(league: str, season: str):
    """
    Scrapes shot-level data from matches and saves raw API payloads per match.

    Args:
        league (str): The league code (e.g., 'EPL')
        season (str): The season year (e.g., '2025')
    """
    output_root = f"{league}_{season}"
    os.makedirs(output_root, exist_ok=True)

    with UnderstatClient() as understat:
        fixtures = understat.league(league=league).get_match_data(season=season)
        match_ids = [int(match['id']) for match in fixtures if match.get('isResult') is True]
        print(f'Found {len(match_ids)} completed {league} matches for season {season}.')

        for match_id in tqdm(match_ids, desc='Downloading Match Data'):
            match_dir = os.path.join(output_root, str(match_id))
            os.makedirs(match_dir, exist_ok=True)

            try:
                match_endpoint = understat.match(match=str(match_id))
                raw_shots = match_endpoint.get_shot_data(timeout=30)
                raw_context = match_endpoint.get_match_info(timeout=30)

                shots_file = os.path.join(match_dir, 'shots.json')
                context_file = os.path.join(match_dir, 'context.json')

                if not os.path.exists(shots_file):
                    with open(shots_file, 'w', encoding='utf-8') as f:
                        json.dump(raw_shots, f, indent=2, default=str)
                if not os.path.exists(context_file):
                    with open(context_file, 'w', encoding='utf-8') as f:
                        json.dump(raw_context, f, indent=2, default=str)

                # print(f'Saved match {match_id} data to {match_dir}.')

            except Exception as exc:
                print(f"\n[Warning] Could not save data for Match ID {match_id}: {exc}")

            time.sleep(0.4)


scrape_match_data(league='EPL', season='2025')

Found 370 completed EPL matches for season 2025.
